## BiLSTM-CRF-LSTM
Word-level BiLSTM + CRF with character-level LSTM embeddings.

### Load data

In [2]:
import numpy as np
import pickle
import json
import random
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')

Device  : cpu
PyTorch : 2.12.0+cpu


In [3]:
DATA_DIR  = Path('../../datasets/processed_data')
SPLIT_DIR = Path('../../datasets/split_data')

X_tokens_train = np.load(DATA_DIR / 'X_tokens_train.npy')
X_chars_train  = np.load(DATA_DIR / 'X_chars_train.npy')
Y_labels_train = np.load(DATA_DIR / 'Y_labels_train.npy')
masks_train    = np.load(DATA_DIR / 'masks_train.npy')

X_tokens_val   = np.load(DATA_DIR / 'X_tokens_val.npy')
X_chars_val    = np.load(DATA_DIR / 'X_chars_val.npy')
Y_labels_val   = np.load(DATA_DIR / 'Y_labels_val.npy')
masks_val      = np.load(DATA_DIR / 'masks_val.npy')

X_tokens_test  = np.load(DATA_DIR / 'X_tokens_test.npy')
X_chars_test   = np.load(DATA_DIR / 'X_chars_test.npy')
Y_labels_test  = np.load(DATA_DIR / 'Y_labels_test.npy')
masks_test     = np.load(DATA_DIR / 'masks_test.npy')

emb_mat = np.load(DATA_DIR / 'embedding_matrix.npy')

with open(DATA_DIR / 'vocabs.pkl', 'rb') as f:
    vocabs = pickle.load(f)

token2id      = vocabs['token2id']
id2token      = vocabs['id2token']
label2id      = vocabs['label2id']
id2label      = vocabs['id2label']
char2id       = vocabs['char2id']
PAD_ID        = vocabs['PAD_ID']
UNK_ID        = vocabs['UNK_ID']
PAD_LABEL_ID  = vocabs['PAD_LABEL_ID']
Entity_labels = vocabs['Entity_labels']
MAX_LEN       = vocabs['MAX_LEN_LSTM']
EMBED_DIM     = vocabs['EMBED_DIM']

NUM_LABELS = len(label2id)
CHAR_SIZE  = len(char2id)
MAX_CHAR   = X_chars_train.shape[2]

with open(SPLIT_DIR / 'split_indices.json') as f:
    split_idx = json.load(f)
idx_train = split_idx['idx_train']
idx_val   = split_idx['idx_val']
idx_test  = split_idx['idx_test']
idx_train_balanced = split_idx['idx_train_balanced']

print(f'Train / Val / Test : {len(idx_train)} / {len(idx_val)} / {len(idx_test)}')
print(f'Train balanced     : {len(idx_train_balanced)}')
print(f'X_tokens_train     : {X_tokens_train.shape}')
print(f'X_chars_train      : {X_chars_train.shape}')
print(f'Vocab size         : {len(token2id):,}')
print(f'Char vocab         : {CHAR_SIZE}')
print(f'Num labels         : {NUM_LABELS}')

Train / Val / Test : 154 / 33 / 33
Train balanced     : 300
X_tokens_train     : (300, 256)
X_chars_train      : (300, 256, 20)
Vocab size         : 3,378
Char vocab         : 59
Num labels         : 22


### Dataset & DataLoader

In [4]:
class ResumeNERDataset(Dataset):
    def __init__(self, X_tokens, X_chars, Y_labels, masks):
        self.X_tokens = torch.tensor(X_tokens, dtype=torch.long)
        self.X_chars  = torch.tensor(X_chars,  dtype=torch.long)
        self.Y_labels = torch.tensor(Y_labels, dtype=torch.long)
        self.masks    = torch.tensor(masks,    dtype=torch.bool)

    def __len__(self):
        return len(self.X_tokens)

    def __getitem__(self, idx):
        return self.X_tokens[idx], self.X_chars[idx], self.Y_labels[idx], self.masks[idx]


BATCH_SIZE = 32

train_ds = ResumeNERDataset(X_tokens_train, X_chars_train, Y_labels_train, masks_train)
val_ds   = ResumeNERDataset(X_tokens_val,   X_chars_val,   Y_labels_val,   masks_val)
test_ds  = ResumeNERDataset(X_tokens_test,  X_chars_test,  Y_labels_test,  masks_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

Train: 300 | Val: 33 | Test: 33


### Model — CharLSTM + BiLSTM + CRF

In [5]:
from torchcrf import CRF


class CharLSTM(nn.Module):
    """
    Character-level LSTM → fixed-size embedding per token.

    Takes the last hidden state of a forward LSTM over character IDs.

    Input  : (batch * seq_len, max_char)  — char IDs per token
    Output : (batch, seq_len, char_hidden_dim)
    """
    def __init__(self, char_vocab_size, char_embed_dim=30, char_hidden_dim=50):
        super().__init__()
        self.embed = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(
            input_size=char_embed_dim,
            hidden_size=char_hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=False,   # forward only — last hidden = char repr
        )
        self.out_dim = char_hidden_dim

    def forward(self, char_ids):
        B, L, C = char_ids.shape
        x = char_ids.view(B * L, C)              # (B*L, C)
        x = self.embed(x)                        # (B*L, C, E_c)
        _, (h, _) = self.lstm(x)                 # h: (1, B*L, char_hidden)
        return h.squeeze(0).view(B, L, -1)       # (B, L, char_hidden)


class BiLSTM_CRF(nn.Module):
    """
    BiLSTM-CRF with CharLSTM for NER.

    Pipeline
    --------
    Token IDs → FastText word embeddings  ┐
    Char IDs  → CharLSTM char embeddings  ├─ concat → BiLSTM → Linear → CRF
    """

    def __init__(
        self,
        embedding_matrix  : np.ndarray,
        char_vocab_size   : int,
        num_tags          : int,
        pad_token_id      : int,
        pad_tag_id        : int,
        hidden_dim        : int   = 256,
        num_layers        : int   = 2,
        char_embed_dim    : int   = 30,
        char_hidden_dim   : int   = 50,
        dropout           : float = 0.5,
        freeze_embeddings : bool  = False,
    ):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape

        self.word_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)
        self.word_emb.weight = nn.Parameter(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            requires_grad=not freeze_embeddings
        )

        self.char_lstm = CharLSTM(char_vocab_size, char_embed_dim, char_hidden_dim)
        self.dropout   = nn.Dropout(dropout)

        lstm_input = embed_dim + char_hidden_dim
        self.lstm = nn.LSTM(
            input_size=lstm_input,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.fc  = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        self.pad_tag_id = pad_tag_id

    def _get_emissions(self, token_ids, char_ids, mask):
        word_x = self.dropout(self.word_emb(token_ids))    # (B, L, E)
        char_x = self.dropout(self.char_lstm(char_ids))    # (B, L, char_hidden)
        x = torch.cat([word_x, char_x], dim=-1)            # (B, L, E+char_hidden)
        x, _ = self.lstm(x)
        x = self.dropout(x)
        return self.fc(x)                                   # (B, L, T)

    def forward(self, token_ids, char_ids, labels, mask):
        emissions = self._get_emissions(token_ids, char_ids, mask)
        return -self.crf(emissions, labels, mask=mask, reduction='mean')

    @torch.no_grad()
    def decode(self, token_ids, char_ids, mask):
        emissions = self._get_emissions(token_ids, char_ids, mask)
        return self.crf.decode(emissions, mask=mask)

### Instantiate model

In [6]:
model = BiLSTM_CRF(
    embedding_matrix  = emb_mat,
    char_vocab_size   = CHAR_SIZE,
    num_tags          = NUM_LABELS,
    pad_token_id      = PAD_ID,
    pad_tag_id        = PAD_LABEL_ID,
    hidden_dim        = 256,
    num_layers        = 2,
    char_embed_dim    = 30,
    char_hidden_dim   = 50,
    dropout           = 0.5,
    freeze_embeddings = False,
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTrainable params : {trainable:,}')
print(f'Total params     : {total:,}')

BiLSTM_CRF(
  (word_emb): Embedding(3378, 300, padding_idx=0)
  (char_lstm): CharLSTM(
    (embed): Embedding(59, 30, padding_idx=0)
    (lstm): LSTM(30, 50, batch_first=True)
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (lstm): LSTM(350, 128, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=256, out_features=22, bias=True)
  (crf): CRF(num_tags=22)
)

Trainable params : 1,924,536
Total params     : 1,924,536


### Training helpers

In [7]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for token_ids, char_ids, labels, mask in loader:
        token_ids = token_ids.to(device)
        char_ids  = char_ids.to(device)
        labels    = labels.to(device)
        mask      = mask.to(device)
        optimizer.zero_grad()
        loss = model(token_ids, char_ids, labels, mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0
    for token_ids, char_ids, labels, mask in loader:
        token_ids = token_ids.to(device)
        char_ids  = char_ids.to(device)
        labels    = labels.to(device)
        mask      = mask.to(device)
        loss = model(token_ids, char_ids, labels, mask)
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def collect_predictions(model, loader, device, id2label, pad_label_id):
    model.eval()
    y_true, y_pred = [], []
    for token_ids, char_ids, labels, mask in loader:
        token_ids = token_ids.to(device)
        char_ids  = char_ids.to(device)
        mask_bool = mask.bool().to(device)
        preds = model.decode(token_ids, char_ids, mask_bool)
        for i, length in enumerate(mask.sum(dim=1).tolist()):
            length = int(length)
            y_true.extend(labels[i, :length].tolist())
            y_pred.extend(preds[i][:length])
    y_true = [id2label.get(t, 'O') for t in y_true]
    y_pred = [id2label.get(t, 'O') for t in y_pred]
    return y_true, y_pred

### Random search

In [8]:
PARAM_DISTRIBUTIONS = {
    'hidden_dim'       : [128, 256, 512],
    'num_layers'       : [1, 2],
    'char_embed_dim'   : [25, 30, 50],
    'char_hidden_dim'  : [50, 100],
    'dropout'          : [0.3, 0.5],
    'freeze_embeddings': [False],
    'lr'               : [1e-3, 3e-3, 5e-4],
    'weight_decay'     : [0.0, 1e-4, 5e-4],
    'batch_size'       : [16, 32],
}

N_TRIALS      = 10
SEARCH_EPOCHS = 20
PATIENCE      = 3
Path('model_result').mkdir(exist_ok=True)
SEARCH_CKPT   = 'model_result/bilstm_crf_lstm_search.pt'


def run_trial(params, trial_id):
    t_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True)
    v_loader = DataLoader(val_ds,   batch_size=params['batch_size'], shuffle=False)

    m = BiLSTM_CRF(
        embedding_matrix  = emb_mat,
        char_vocab_size   = CHAR_SIZE,
        num_tags          = NUM_LABELS,
        pad_token_id      = PAD_ID,
        pad_tag_id        = PAD_LABEL_ID,
        hidden_dim        = params['hidden_dim'],
        num_layers        = params['num_layers'],
        char_embed_dim    = params['char_embed_dim'],
        char_hidden_dim   = params['char_hidden_dim'],
        dropout           = params['dropout'],
        freeze_embeddings = params['freeze_embeddings'],
    ).to(DEVICE)

    opt = optim.Adam(m.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    best_val, no_imp = float('inf'), 0

    for ep in range(1, SEARCH_EPOCHS + 1):
        train_epoch(m, t_loader, opt, DEVICE)
        val_loss = eval_epoch(m, v_loader, DEVICE)
        if val_loss < best_val:
            best_val = val_loss
            torch.save(m.state_dict(), SEARCH_CKPT)
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                break

    print(f'Trial {trial_id:2d} | val_loss={best_val:.4f} | {params}')
    return {'trial': trial_id, 'val_loss': best_val, **params}


random.seed(42)
results = []
for i in range(1, N_TRIALS + 1):
    params = {k: random.choice(v) for k, v in PARAM_DISTRIBUTIONS.items()}
    results.append(run_trial(params, i))

results_df = pd.DataFrame(results).sort_values('val_loss').reset_index(drop=True)
print('\nTop 5:')
print(results_df.head())

Trial  1 | val_loss=37.5198 | {'hidden_dim': 512, 'num_layers': 1, 'char_embed_dim': 25, 'char_hidden_dim': 100, 'dropout': 0.3, 'freeze_embeddings': False, 'lr': 0.001, 'weight_decay': 0.0005, 'batch_size': 16}
Trial  2 | val_loss=38.7740 | {'hidden_dim': 512, 'num_layers': 1, 'char_embed_dim': 50, 'char_hidden_dim': 100, 'dropout': 0.3, 'freeze_embeddings': False, 'lr': 0.001, 'weight_decay': 0.0, 'batch_size': 16}
Trial  3 | val_loss=36.9331 | {'hidden_dim': 512, 'num_layers': 1, 'char_embed_dim': 50, 'char_hidden_dim': 50, 'dropout': 0.5, 'freeze_embeddings': False, 'lr': 0.003, 'weight_decay': 0.0005, 'batch_size': 32}
Trial  4 | val_loss=43.1938 | {'hidden_dim': 128, 'num_layers': 1, 'char_embed_dim': 50, 'char_hidden_dim': 100, 'dropout': 0.5, 'freeze_embeddings': False, 'lr': 0.001, 'weight_decay': 0.0, 'batch_size': 32}
Trial  5 | val_loss=45.3555 | {'hidden_dim': 128, 'num_layers': 1, 'char_embed_dim': 30, 'char_hidden_dim': 50, 'dropout': 0.5, 'freeze_embeddings': False, 'lr

### Retrain best config

In [9]:
best = results_df.iloc[0].to_dict()
print('Best config:')
for k, v in best.items():
    print(f'  {k}: {v}')

CKPT_PATH   = 'model_result/bilstm_crf_lstm.pt'
FULL_EPOCHS = 45
PATIENCE    = 5

model = BiLSTM_CRF(
    embedding_matrix  = emb_mat,
    char_vocab_size   = CHAR_SIZE,
    num_tags          = NUM_LABELS,
    pad_token_id      = PAD_ID,
    pad_tag_id        = PAD_LABEL_ID,
    hidden_dim        = int(best['hidden_dim']),
    num_layers        = int(best['num_layers']),
    char_embed_dim    = int(best['char_embed_dim']),
    char_hidden_dim   = int(best['char_hidden_dim']),
    dropout           = float(best['dropout']),
    freeze_embeddings = bool(best['freeze_embeddings']),
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=float(best['lr']), weight_decay=float(best['weight_decay']))
t_loader  = DataLoader(train_ds, batch_size=int(best['batch_size']), shuffle=True)
v_loader  = DataLoader(val_ds,   batch_size=int(best['batch_size']), shuffle=False)

best_val, no_imp = float('inf'), 0
train_losses, val_losses = [], []

for ep in range(1, FULL_EPOCHS + 1):
    tr = train_epoch(model, t_loader, optimizer, DEVICE)
    vl = eval_epoch(model, v_loader, DEVICE)
    train_losses.append(tr)
    val_losses.append(vl)
    mark = ''
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), CKPT_PATH)
        no_imp = 0
        mark = ' *'
    else:
        no_imp += 1
    print(f'Epoch {ep:3d}/{FULL_EPOCHS}  train={tr:.4f}  val={vl:.4f}{mark}')
    if no_imp >= PATIENCE:
        print('Early stopping.')
        break

plt.figure(figsize=(8,4))
plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('BiLSTM-CRF-LSTM Loss')
plt.tight_layout(); plt.savefig('model_result/bilstm_crf_lstm_loss.png', dpi=100); plt.show()

Best config:
  trial: 7
  val_loss: 36.60067621866862
  hidden_dim: 512
  num_layers: 1
  char_embed_dim: 50
  char_hidden_dim: 50
  dropout: 0.3
  freeze_embeddings: False
  lr: 0.003
  weight_decay: 0.0
  batch_size: 16
Epoch   1/45  train=201.7713  val=82.8729 *
Epoch   2/45  train=83.8744  val=54.1681 *
Epoch   3/45  train=45.0017  val=39.4512 *
Epoch   4/45  train=26.6483  val=37.0757 *
Epoch   5/45  train=17.3330  val=37.9053
Epoch   6/45  train=12.3943  val=44.0033
Epoch   7/45  train=9.3136  val=43.0911
Epoch   8/45  train=7.1697  val=47.4206
Epoch   9/45  train=5.0150  val=49.8472
Early stopping.


### Evaluation

In [10]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

y_true, y_pred = collect_predictions(model, test_loader, DEVICE, id2label, PAD_LABEL_ID)

correct = sum(t == p for t, p in zip(y_true, y_pred))
print(f'Token accuracy : {correct/len(y_true):.4f}  ({correct}/{len(y_true)})')

labels_to_report = [l for l in set(y_true) if l not in ('O', '<PAD_LABEL>')]
print()
print(classification_report(y_true, y_pred, labels=labels_to_report, zero_division=0))

Token accuracy : 0.9097  (6313/6940)

                       precision    recall  f1-score   support

        I-Designation       0.83      0.38      0.52       114
      I-Email Address       0.90      0.82      0.86        11
I-Companies worked at       0.70      0.19      0.30        37
             I-Skills       0.59      0.39      0.47       345
       B-College Name       0.57      0.63      0.60        19
      B-Email Address       0.73      0.92      0.81        26
B-Years of Experience       0.00      0.00      0.00         1
               B-Name       1.00      0.97      0.99        34
           I-Location       0.00      0.00      0.00         1
    B-Graduation Year       0.75      0.14      0.24        21
             B-Skills       0.60      0.12      0.19        26
           B-Location       0.00      0.00      0.00         8
        B-Designation       0.74      0.42      0.54        66
               I-Name       0.82      0.97      0.89        33
             I-D

In [11]:
try:
    from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1

    def to_seqeval(flat_tags, masks_arr):
        seqs, idx = [], 0
        for m in masks_arr:
            length = int(m.sum())
            seqs.append(flat_tags[idx:idx+length])
            idx += length
        return seqs

    y_true_seq = to_seqeval(y_true, masks_test)
    y_pred_seq = to_seqeval(y_pred, masks_test)

    print('=== Test Set Evaluation (seqeval — entity-level) ===')
    print(seq_report(y_true_seq, y_pred_seq, zero_division=0))
    test_f1 = seq_f1(y_true_seq, y_pred_seq, zero_division=0)
    print(f'Test F1 (seqeval) : {test_f1:.4f}')
except ImportError:
    print('seqeval not installed — pip install seqeval')

=== Test Set Evaluation (seqeval — entity-level) ===
                     precision    recall  f1-score   support

       College Name       0.23      0.26      0.24        19
Companies worked at       0.43      0.42      0.43        52
             Degree       0.50      0.48      0.49        21
        Designation       0.51      0.30      0.38        66
      Email Address       0.70      0.88      0.78        26
    Graduation Year       0.75      0.14      0.24        21
           Location       0.00      0.00      0.00         8
               Name       0.79      0.79      0.79        34
             Skills       0.11      0.07      0.09        27
Years of Experience       0.00      0.00      0.00         1

          micro avg       0.50      0.41      0.45       275
          macro avg       0.40      0.34      0.34       275
       weighted avg       0.49      0.41      0.43       275

Test F1 (seqeval) : 0.4498


### Save

In [12]:
results_out = {
    'model_name'   : 'BiLSTM-CRF-LSTM',
    'best_params'  : best,
    'train_losses' : train_losses,
    'val_losses'   : val_losses,
    'y_true'       : y_true,
    'y_pred'       : y_pred,
}

with open('model_result/results_bilstm_crf_lstm.pkl', 'wb') as f:
    pickle.dump(results_out, f)

print('Model saved   -> model_result/bilstm_crf_lstm.pt')
print('Results saved -> model_result/results_bilstm_crf_lstm.pkl')

Model saved   -> model_result/bilstm_crf_lstm.pt
Results saved -> model_result/results_bilstm_crf_lstm.pkl
